# 04 · Training on Kaggle: the cloud teacher, then the edge student

**One notebook, one Kaggle session (T4 GPU, max 12 h):**
- **Part A — teacher** (PyTorch): TI-STGNN, its ablation runs, benchmarks, out-of-fold outputs for the student
- **Part B — student** (Keras): GDC-CNN, tuning, distillation S0-S4, ablation, benchmarks, export for the ESP32

Time plan: Part A starts no new run after 8.5 h, Part B none after 11.3 h; the export at the end always runs.

# Part A · Cloud teacher: TI-STGNN

The teacher sees **all stations at once** and knows where the wind blows. It is built from 5 blocks:

```
all stations x 24 h x (23 sensor features + 7 ERA5 weather)
  1. Embedding                       turn each hour's numbers into a vector
  2. Transformer over time           each station looks at its own last 24 h        -> feat_seq, attention
  3. Wind graph attention (GAT)      each station listens to its UPWIND neighbours   -> h_spatial
  4. ADR physics                     advection + diffusion + reaction between stations -> h_adr
  5. Decoder + future weather        PM2.5 / PM10 change for +1 ... +6 h
```

**Fair comparison:** the teacher's station inputs are the same simulated sensor features as the student's.
Its advantage is the network (neighbours, wind, physics), never cleaner PM.

Papers: TransNet (Dimri et al. 2026) for the wind graph and ADR, WSDST-GAT (Wu et al. 2026) for GAT + Transformer
+ decoder, Wang et al. 2024 for future weather in the decoder.

**What Part A produces** (in `/kaggle/working`):

| file | what |
|---|---|
| `teacher_benchmarks.csv`, `teacher_summary.csv` | every benchmark of every run (section 9) |
| `teacher_T0.npz` … `teacher_T3.npz` | out-of-fold teacher outputs, needed by the student (Part B) |
| `teacher_best.pt` + `teacher_best.json` | the best teacher, for live cloud inference later |

## A0. Setup

In [ ]:
# imports (Part B adds TensorFlow / Keras)
import copy
import json
import math
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F

In [ ]:
# where the data is (Kaggle or laptop) and where results go
ON_KAGGLE = Path("/kaggle/input").exists()
if ON_KAGGLE:
    FEAT = next(Path("/kaggle/input").rglob("grid_u0.npz")).parent
    OUT = Path("/kaggle/working")
else:
    FEAT = Path("../data/4_features")
    OUT = Path("../data/5_models/teacher")
OUT.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("data:", FEAT, "| device:", DEVICE)
NOTEBOOK_START = time.time()           # both parts share one 12-hour Kaggle session

In [ ]:
# settings (SMOKE = tiny quick run to test the notebook; the real run is on Kaggle)
SMOKE = not ON_KAGGLE
SETTINGS = dict(
    d_model=64, heads=4, layers=2, dropout=0.1,            # size of the model
    d_min_km=1.0, calm_ms=0.3, sigma_km=10.0,              # graph
    adr_steps=2, adr_h=0.1,                                 # physics block
    batch=64, lr=1e-3, weight_decay=1e-4,                   # training (batch 64 keeps the GPU busy)
    samples_per_epoch=20_000, max_epochs=40, patience=6,
    station_dropout=0.1, pinball_weight=0.5, folds=3,
)
if SMOKE:
    SETTINGS.update(samples_per_epoch=64, max_epochs=1, folds=2)
H = 6                           # the teacher forecasts 6 hours ahead
USE_AMP = DEVICE == "cuda"      # mixed precision: float16 maths on the T4's tensor cores (about 2x faster)
TEACHER_BUDGET_H = 8.5          # no new teacher run after 8.5 h (Part B needs ~3 h, Kaggle stops at 12 h)
SETTINGS

**The runs (ablation study).** Each run switches blocks on or off:

| run | graph | wind | ADR | future weather | question |
|---|---|---|---|---|---|
| T0 | – | – | – | ✓ | how good is "time only"? |
| T1 | ✓ | – | – | ✓ | do neighbours help? |
| T2 | ✓ | ✓ | – | ✓ | does knowing the wind help? |
| T3 | ✓ | ✓ | ✓ | ✓ | does transport physics help? (the full teacher) |
| T3-no-wind | ✓ | – | ✓ | ✓ | the full teacher without the wind |
| T3-no-future | ✓ | ✓ | ✓ | – | the full teacher without ERA5 "future weather" (how much the oracle weather helps) |
| T3-seed1, T3-seed2 | ✓ | ✓ | ✓ | ✓ | T3 again from another random start: is a difference real or luck? |

- T0 → T3 **add** one block at a time; the "no-" runs **remove** one block from the full teacher.
- T3 without ADR is T2. T3 without the graph cannot exist: ADR moves pollution along the graph.
- The student (Part B) needs out-of-fold outputs of T0-T3 only.
- Runs go in this order; if the 8.5-hour budget runs out, the last ones are skipped.

In [ ]:
# every run: switches (graph, wind, ADR, future weather), random seed, does the student need its outputs?
RUNS = {
    "T0": dict(switches=(False, False, False, True), seed=0, for_student=True),
    "T1": dict(switches=(True, False, False, True), seed=0, for_student=True),
    "T2": dict(switches=(True, True, False, True), seed=0, for_student=True),
    "T3": dict(switches=(True, True, True, True), seed=0, for_student=True),
    "T3-no-wind": dict(switches=(True, False, True, True), seed=0, for_student=False),
    "T3-no-future": dict(switches=(True, True, True, False), seed=0, for_student=False),
    "T3-seed1": dict(switches=(True, True, True, True), seed=1, for_student=False),
    "T3-seed2": dict(switches=(True, True, True, True), seed=2, for_student=False),
}
VARIANTS = list(RUNS)           # kaggle_run.py --variants can pick a subset

## A1. Data
`grid_u<k>.npz` (from 03): `X[hour, station, 23 features]`, masks `M`, ERA5 weather `W[hour, station, 7]`,
true PM `pm[hour, station, 2]` (targets only), and station info. One grid per virtual unit (3 "versions" of the network).

In [ ]:
# load the 3 grids
grids = {u: dict(np.load(FEAT / f"grid_u{u}.npz")) for u in range(3)}
g0 = grids[0]
stations = list(g0["stations"])
print("hours:", g0["X"].shape[0], "| stations:", len(stations), "| features:", g0["X"].shape[2])

In [ ]:
# check: one row per hour, in order -> "past" = rows before t, "future" = rows after t
assert (np.diff(g0["ts"]) == 3600).all()
MONTH = pd.to_datetime(g0["ts"], unit="s", utc=True).tz_convert("Asia/Kolkata").month.to_numpy()   # for seasons

In [ ]:
# training stations (27) and all stations (32 = + 5 never-seen stations for the final test)
train_idx = np.flatnonzero(~g0["holdout"])
all_idx = np.arange(len(stations))
HOLDOUT_COLS = np.flatnonzero(g0["holdout"])          # their columns when we predict all 32 stations
lat, lon = g0["static"][:, 0], g0["static"][:, 1]
len(train_idx), len(all_idx)

In [ ]:
# mean / std of the 30 inputs (23 sensor + 7 weather), from training hours and stations only
train_hours = np.flatnonzero(g0["split"] == "train")
x_train = np.concatenate([g0["X"][train_hours][:, train_idx], g0["W"][train_hours][:, train_idx]], axis=2).reshape(-1, 30)
m_train = np.concatenate([g0["M"][train_hours][:, train_idx], np.ones_like(g0["W"][train_hours][:, train_idx], np.uint8)], axis=2).reshape(-1, 30)
x_train = np.where(m_train > 0, x_train, np.nan)
MEAN, STD = np.nanmean(x_train, axis=0), np.maximum(np.nanstd(x_train, axis=0), 1e-3)
STATIC = (g0["static"] - g0["static"].mean(0)) / np.maximum(g0["static"].std(0), 1e-3)
del x_train, m_train

In [ ]:
# put the 3 grids on the GPU once (~1.1 GB of the T4's 16 GB): cutting batches there is much faster than in numpy
def on_gpu(key, dtype=torch.float32):
    return torch.tensor(np.stack([grids[u][key] for u in range(3)]), dtype=dtype, device=DEVICE)   # [unit, hour, station, ...]


GX, GM, GW = on_gpu("X"), on_gpu("M", torch.uint8), on_gpu("W")
LOG_PM = torch.log1p(on_gpu("pm"))                     # targets only, never an input
IST_HOUR = torch.tensor(g0["ist_hour"], device=DEVICE)
MEAN_T, STD_T = torch.tensor(MEAN, dtype=torch.float32, device=DEVICE), torch.tensor(STD, dtype=torch.float32, device=DEVICE)
STATIC_T = torch.tensor(STATIC, dtype=torch.float32, device=DEVICE)
TRAIN_IDX, ALL_IDX = torch.tensor(train_idx, device=DEVICE), torch.tensor(all_idx, device=DEVICE)
if DEVICE == "cuda":
    print(f"GPU memory used by the grids: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## A2. One graph sample = all stations at one hour $t$
| input | shape | what |
|---|---|---|
| X, M | [N, 24, 30] | last 24 h: 23 sensor features + 7 ERA5 weather (z-scored), and masks |
| S | [N, 6] | lat, lon, elevation, ... (z-scored) |
| X_fut | [N, 6, 9] | ERA5 weather at t+1 … t+6 + hour of day (sin, cos) |
| wind_t | [N, 2] | wind speed, direction FROM (deg) now |
| y, y_mask | [N, 12] | targets $\ln(1+PM_{t+h}) - \ln(1+PM_t)$, h = 1…6, PM2.5 then PM10 |
| base_pm | [N, 2] | true PM now (only to turn errors into µg/m³ in the benchmarks) |

A batch is many samples at once: `pairs = [[unit, hour], ...]`, all cut on the GPU.
Note: the "future weather" is ERA5 actual weather (old weather forecasts are not archived) → a limitation we report.

In [ ]:
# cut a batch of samples out of the GPU grids. pairs = tensor [[unit, hour], ...], idx = which stations
def make_batch(pairs, idx, drop=0.0):
    u, t = pairs[:, 0], pairs[:, 1]
    past = t[:, None] + torch.arange(-23, 1, device=DEVICE)        # the last 24 hours (t included)
    future = t[:, None] + torch.arange(1, H + 1, device=DEVICE)    # t+1 ... t+6
    B, N = len(t), len(idx)

    def cut(grid, hours):                                          # -> [batch, station, hour, channel]
        return grid[u[:, None], hours][:, :, idx].transpose(1, 2)

    # inputs: 23 sensor features + 7 weather, z-scored; missing values -> 0 with mask 0
    x = torch.cat([cut(GX, past), cut(GW, past)], dim=-1)
    m = torch.cat([cut(GM, past).float(), torch.isfinite(cut(GW, past)).float()], dim=-1) * torch.isfinite(x)
    x = torch.where(m > 0, (x - MEAN_T) / STD_T, 0.0)

    # future weather + clock of the 6 forecast hours (for the decoder)
    angle = 2 * math.pi * IST_HOUR[future] / 24
    clock = torch.stack([torch.sin(angle), torch.cos(angle)], dim=-1)[:, None].expand(B, N, H, 2)
    weather = (cut(GW, future) - MEAN_T[23:]) / STD_T[23:]
    x_fut = torch.nan_to_num(torch.cat([weather, clock], dim=-1))

    # wind now (speed, direction it comes FROM), from the weather data
    wind_u, wind_v = torch.nan_to_num(GW[u, t][:, idx, 3]), torch.nan_to_num(GW[u, t][:, idx, 4])
    wind_t = torch.stack([torch.hypot(wind_u, wind_v), torch.rad2deg(torch.atan2(-wind_u, -wind_v)) % 360], dim=-1)

    # targets: change of ln(1+PM) after 1..6 h, ordered [pm25 h1..h6, pm10 h1..h6]
    log_pm_now = LOG_PM[u, t][:, idx]                                               # [B, N, 2]
    y = (cut(LOG_PM, future) - log_pm_now[:, :, None]).transpose(2, 3).reshape(B, N, 2 * H)

    if drop > 0:                                                   # hide random stations (makes the model robust)
        keep = (torch.rand(B, N, 1, 1, device=DEVICE) >= drop).float()
        x, m = x * keep, m * keep
    return dict(X=x, M=m, S=STATIC_T[idx].expand(B, -1, -1), X_fut=x_fut, wind_t=wind_t,
                y=torch.nan_to_num(y), y_mask=torch.isfinite(y).float(), base_pm=torch.expm1(log_pm_now))

In [ ]:
# all (unit, hour) pairs of a split where the window fits and at least one station has a target
def sample_pairs(split, idx):
    pairs = []
    for u, g in grids.items():
        T = len(g["ts"])
        t = np.arange(23, T - H)
        same_split = (g["split"][t] == split) & (g["split"][t + H] == split)
        log_pm = np.log1p(g["pm"][:, idx])
        has_target = np.zeros(T, bool)
        for h in range(1, H + 1):
            has_target[: T - h] |= np.isfinite(log_pm[h:] - log_pm[:-h]).any(axis=(1, 2))
        pairs += [(u, int(x)) for x in t[same_split & has_target[t]]]
    return torch.tensor(pairs, device=DEVICE)


train_pairs, val_pairs = sample_pairs("train", train_idx), sample_pairs("val", train_idx)
len(train_pairs), len(val_pairs)

In [ ]:
# look at the shapes of one batch of 2 samples
{k: tuple(v.shape) for k, v in make_batch(train_pairs[5000:5002], TRAIN_IDX).items()}

## A3. The graph

**Distance and bearing** between stations (great circle):
$d_{ij}$ in km, $\theta_{ij}$ = compass direction from station $i$ to station $j$.

**Wind prior** (TransNet): how strongly the wind at station $i$ carries air to station $j$
$$w_{ij} = \frac{\mathrm{ReLU}\big(ws_i\cos(\theta_{ij} - \varphi_i)\big)}{\max(d_{ij},\ 1\,km)},\qquad \varphi_i = wd_i^{from} + 180°$$
- $\varphi_i$ = direction the air moves **to** (TransNet puts the "from" direction there, which points upwind — we fix that).
- Calm air (< 0.3 m/s) carries nothing. For T1 (no wind) we use $w_{ij} = 1/d_{ij}$.

In [ ]:
# distance (km) and bearing (deg) between every pair of stations
def geometry(lat, lon):
    la, lo = np.deg2rad(lat)[:, None], np.deg2rad(lon)[:, None]
    a = np.sin((la.T - la) / 2) ** 2 + np.cos(la) * np.cos(la.T) * np.sin((lo.T - lo) / 2) ** 2
    dist = 2 * 6371 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    y = np.sin(lo.T - lo) * np.cos(la.T)
    x = np.cos(la) * np.sin(la.T) - np.sin(la) * np.cos(la.T) * np.cos(lo.T - lo)
    bearing = np.rad2deg(np.arctan2(y, x)) % 360
    return torch.tensor(dist, dtype=torch.float32, device=DEVICE), torch.tensor(bearing, dtype=torch.float32, device=DEVICE)

In [ ]:
# wind prior w[b, i, j]: air from station i to station j (formula above)
def wind_prior(ws, wd_from, dist, bearing):
    toward = (wd_from + 180) % 360
    angle = torch.deg2rad(bearing[None] - toward[:, :, None])
    w = torch.relu(ws[:, :, None] * torch.cos(angle)) / dist.clamp_min(SETTINGS["d_min_km"])[None]
    w = w * (ws >= SETTINGS["calm_ms"]).float()[:, :, None]
    return w * (1 - torch.eye(dist.shape[0], device=DEVICE))


# T1: no wind, closer stations simply matter more
def distance_prior(dist):
    return (1 / dist.clamp_min(SETTINGS["d_min_km"])) * (1 - torch.eye(dist.shape[0], device=DEVICE))

In [ ]:
# check: wind from the west -> the eastern neighbour gets weight, the western one nothing
d, b = geometry(np.array([28.6, 28.6, 28.6]), np.array([76.9, 77.0, 77.1]))    # west, middle, east
w = wind_prior(torch.full((1, 3), 3.0, device=DEVICE), torch.full((1, 3), 270.0, device=DEVICE), d, b)[0]
print(f"middle -> east: {w[1, 2]:.3f}   middle -> west: {w[1, 0]:.3f}")
assert w[1, 2] > 0 and w[1, 0] == 0

In [ ]:
# draw the stations and the wind arrows of one hour
dist, bearing = geometry(lat[train_idx], lon[train_idx])
t = int(train_hours[3000])
plt.figure(figsize=(6, 6))
plt.scatter(lon, lat, c=np.where(g0["holdout"], "tab:red", "tab:blue"), s=40)
plt.quiver(lon, lat, g0["W"][t, :, 3], g0["W"][t, :, 4])
plt.title("stations (red = never trained on) and the wind of one hour")
plt.show()

## A4. Model blocks

### 4.1 Embedding
Every hour of every station: $e = W_v[\,x \odot m,\ m\,] + W_s\,s$ — the mask tells the model "0 because missing" apart from a real 0.

In [ ]:
# embedding: values + masks + static info -> vector of size D
class Embedding(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.values = nn.Linear(2 * 30, d)
        self.static = nn.Linear(6, d)

    def forward(self, x, m, s):                       # x, m: [B, N, 24, 30], s: [B, N, 6]
        return self.values(torch.cat([x * m, m], dim=-1)) + self.static(s)[:, :, None, :]

### 4.2 Transformer over time (each station on its own)
Standard pre-norm Transformer layer: $x \leftarrow x + \text{Attention}(\text{LN}(x))$, $x \leftarrow x + \text{FFN}(\text{LN}(x))$,
with sine/cosine position codes. We keep the **attention map** — the student copies it later (Na et al.: $L_{AFD}$).

In [ ]:
# one Transformer layer that also returns its attention map
class TimeLayer(nn.Module):
    def __init__(self, d, heads, dropout):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Dropout(dropout), nn.Linear(4 * d, d))

    def forward(self, x):
        h = self.norm1(x)
        a, attention = self.attn(h, h, h, need_weights=True, average_attn_weights=False)
        x = x + a
        x = x + self.ffn(self.norm2(x))
        return x, attention

In [ ]:
# position codes: hour 0 ... 23 as sines and cosines
def position_codes(length, d):
    pos = torch.arange(length, dtype=torch.float32)[:, None]
    freq = torch.exp(torch.arange(0, d, 2, dtype=torch.float32) * (-math.log(10000.0) / d))
    pe = torch.zeros(length, d)
    pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * freq), torch.cos(pos * freq)
    return pe

### 4.3 Wind graph attention (GAT)
Station $i$ (receiver) mixes in information from stations $j$ (senders):
$$\text{score}_{ij} = \text{LeakyReLU}(a_r^\top W h_i + a_s^\top W h_j) + \beta \log(w_{j \to i} + 10^{-3})$$
$$\alpha_{ij} = \text{softmax}_j(\text{score}_{ij}),\qquad h_i^{spatial} = h_i + W_o \sum_j \alpha_{ij} W h_j$$
Careful: the prior is the edge **from** sender $j$ **to** receiver $i$, i.e. $w_{j\to i}$ (the transpose of $w$).
Missing stations send nothing. $\alpha$ is learned; the wind only nudges it.

In [ ]:
# wind-biased graph attention
class WindGAT(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.W = nn.Linear(d, d, bias=False)
        self.a_receiver, self.a_sender = nn.Linear(d, 1, bias=False), nn.Linear(d, 1, bias=False)
        self.out = nn.Linear(d, d)
        self.beta = nn.Parameter(torch.tensor(1.0))

    def forward(self, h, prior, valid):               # h: [B, N, D], prior: [B, N, N], valid: [B, N]
        n = h.shape[1]
        eye = torch.eye(n, dtype=torch.bool, device=h.device)[None]
        wh = self.W(h)
        score = F.leaky_relu(self.a_receiver(wh) + self.a_sender(wh).transpose(1, 2), 0.2)   # [B, receiver, sender]
        incoming = prior.transpose(1, 2)                                                     # w[j -> i]
        incoming = torch.where(eye, incoming.amax(-1, keepdim=True).expand_as(incoming), incoming)   # self-loop
        score = score + self.beta * torch.log(incoming + 1e-3)
        allowed = valid[:, None, :].bool() | eye
        alpha = torch.softmax(score.masked_fill(~allowed, float("-inf")), dim=-1)
        return h + self.out(alpha @ wh), score

### 4.4 ADR physics (TransNet): K small steps of size $h$
**Advection** (wind carries pollution). $P_{ij}$ = share of station $j$'s state that moves to $i$: softmax of the
GAT scores over the **receivers**, so every station sends out exactly 100 % — mass is conserved:
$$U_i \leftarrow U_i + h\Big(\sum_j P_{ij} U_j - U_i\sum_k P_{ki}\Big)$$
**Diffusion** (pollution spreads to close stations), on the graph $A_{ij} = e^{-d_{ij}^2/\sigma^2}$ with Laplacian
$\hat L = I - D^{-1/2}AD^{-1/2} = V\Lambda V^\top$, solved exactly per channel $c$ with $\kappa_c \in [0, 1]$:
$$U \leftarrow V\,\text{diag}\Big(\frac{1}{1 + h\,\kappa_c\,\lambda}\Big)V^\top U$$
**Reaction** (local change: emissions, chemistry, weather):
$$U \leftarrow U + h\cdot \text{ReLU}\big(R_1 U + \tanh(R_2 U)\odot U + R_3\,\bar h_{time} + R_4\,\text{weather}\big)$$

In [ ]:
# advection weights: softmax over receivers -> every station sends exactly 100 %
def advection_weights(score, valid):
    n = score.shape[-1]
    v = valid.bool()
    allowed = v[:, :, None] & v[:, None, :] & ~torch.eye(n, dtype=torch.bool, device=score.device)[None]
    return torch.nan_to_num(torch.softmax(score.masked_fill(~allowed, float("-inf")), dim=1))

In [ ]:
# diffusion basis: eigen-decomposition of the distance graph's Laplacian (missing stations are cut out)
def diffusion_basis(dist, valid):
    n = dist.shape[0]
    A = torch.exp(-dist**2 / SETTINGS["sigma_km"] ** 2) * (1 - torch.eye(n, device=dist.device))
    A = A[None] * valid[:, :, None] * valid[:, None, :]
    degree = A.sum(-1)
    d_inv_sqrt = torch.where(degree > 0, degree.clamp_min(1e-12).rsqrt(), torch.zeros_like(degree))
    L = torch.diag_embed((degree > 0).float()) - d_inv_sqrt[:, :, None] * A * d_inv_sqrt[:, None, :]
    lam, V = torch.linalg.eigh(L)
    return V, lam.clamp(0, 2)

In [ ]:
# the ADR block
class ADR(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.kappa_raw = nn.Parameter(torch.full((d,), 0.5))
        self.R1 = nn.Sequential(nn.Linear(d, d), nn.LayerNorm(d))
        self.R2, self.R3, self.R4 = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(7, d)

    def forward(self, U, score, V, lam, time_summary, weather, valid):
        P = advection_weights(score, valid)
        kappa = F.hardtanh(self.kappa_raw, 0, 1)                       # no negative diffusion
        h = SETTINGS["adr_h"]
        for _ in range(SETTINGS["adr_steps"]):
            U = U + h * (P @ U - P.sum(1)[:, :, None] * U)              # advection
            U = V @ ((1 / (1 + h * kappa * lam[:, :, None])) * (V.transpose(1, 2) @ U))   # diffusion
            U = U + h * F.relu(self.R1(U) + torch.tanh(self.R2(U)) * U + self.R3(time_summary) + self.R4(weather))  # reaction
        return U

In [ ]:
# check: advection conserves mass
P = advection_weights(torch.randn(2, 5, 5), torch.ones(2, 5))
U = torch.randn(2, 5, 3)
U_next = U + 0.1 * (P @ U - P.sum(1)[:, :, None] * U)
assert torch.allclose(U_next.sum(1), U.sum(1), atol=1e-5)
print("advection keeps the total the same")

### 4.5 Decoder
Fuse the spatial/physics state with the station's own time state: $z = \text{GLU}([h_{adr}, h_t])$.
One query per forecast hour $k$: $q_k = \text{pos}_k + W_f\,\text{weather}_{t+k} + z$.
One Transformer decoder layer (self-attention over the 6 hours, cross-attention to the station's 24 h), then
heads for the forecast and for a q10-q90 band (lower = forecast − softplus(a), upper = forecast + softplus(b)).

In [ ]:
# decoder: 6 forecast hours with future weather
class Decoder(nn.Module):
    def __init__(self, d, heads, dropout):
        super().__init__()
        self.fuse = nn.Linear(2 * d, 2 * d)
        self.hour_code = nn.Embedding(H, d)
        self.weather = nn.Linear(9, d)
        self.layer = nn.TransformerDecoderLayer(d, heads, 4 * d, dropout, batch_first=True, norm_first=True)
        self.forecast = nn.Linear(d, 2)                 # PM2.5, PM10
        self.band = nn.Linear(d, 4)                     # distances to q10 / q90

    def forward(self, h_adr, h_t, feat_seq, x_fut):
        B, N, L, D = feat_seq.shape
        z = F.glu(self.fuse(torch.cat([h_adr, h_t], dim=-1)), dim=-1)
        q = self.hour_code.weight[None, None] + self.weather(x_fut) + z[:, :, None]       # [B, N, 6, D]
        out = self.layer(q.reshape(B * N, H, D), feat_seq.reshape(B * N, L, D)).reshape(B, N, H, D)
        y = self.forecast(out)                                                             # [B, N, 6, 2]
        spread = F.softplus(self.band(out))
        to_12 = lambda t: t.permute(0, 1, 3, 2).reshape(B, N, 2 * H)   # [pm25 h1..h6, pm10 h1..h6]
        return to_12(y), to_12(y - spread[..., :2]), to_12(y + spread[..., 2:])

### 4.6 The whole teacher

In [ ]:
# TI-STGNN: blocks 1-5 wired together; the 4 switches turn blocks on and off (ablation)
class Teacher(nn.Module):
    def __init__(self, use_graph, use_wind, use_adr, use_future):
        super().__init__()
        d = SETTINGS["d_model"]
        self.use_wind, self.use_future = use_wind, use_future
        self.embed = Embedding(d)
        self.register_buffer("pos", position_codes(24, d), persistent=False)
        self.time_layers = nn.ModuleList([TimeLayer(d, SETTINGS["heads"], SETTINGS["dropout"]) for _ in range(SETTINGS["layers"])])
        self.norm = nn.LayerNorm(d)
        self.gat = WindGAT(d) if use_graph else None
        self.adr = ADR(d) if use_adr else None
        self.decoder = Decoder(d, SETTINGS["heads"], SETTINGS["dropout"])

    def forward(self, b, dist, bearing):
        B, N, L, _ = b["X"].shape
        valid = (b["M"][..., :23].sum(dim=(2, 3)) > 0).float()           # station has any sensor data

        # 1-2: each station's own 24 hours
        z = (self.embed(b["X"], b["M"], b["S"]) + self.pos).reshape(B * N, L, -1)
        for layer in self.time_layers:
            z, attention = layer(z)
        feat_seq = self.norm(z).reshape(B, N, L, -1)
        attention = attention.reshape(B, N, -1, L, L)
        h_t = feat_seq[:, :, -1]                                          # state at the current hour

        # 3: listen to the neighbours
        h_spatial, score = h_t, None
        if self.gat is not None:
            ws, wd = b["wind_t"][..., 0], b["wind_t"][..., 1]
            prior = wind_prior(ws, wd, dist, bearing) if self.use_wind else distance_prior(dist)[None].expand(B, -1, -1)
            h_spatial, score = self.gat(h_t, prior, valid)

        # 4: transport physics
        h_adr = h_spatial
        if self.adr is not None:
            with torch.no_grad():
                V, lam = diffusion_basis(dist, valid)
            h_adr = self.adr(h_spatial, score, V, lam, feat_seq.mean(2), b["X"][:, :, -1, 23:], valid)

        # 5: forecast (without future weather the decoder still gets the clock of the 6 hours)
        x_fut = b["X_fut"] if self.use_future else torch.cat([torch.zeros_like(b["X_fut"][..., :7]), b["X_fut"][..., 7:]], dim=-1)
        y, q10, q90 = self.decoder(h_adr, h_t, feat_seq, x_fut)
        return dict(y=y, q10=q10, q90=q90, feat_seq=feat_seq, attention=attention, h_spatial=h_spatial, h_adr=h_adr)

In [ ]:
# check: every run builds and gives 12 outputs per station
batch = make_batch(train_pairs[:2], TRAIN_IDX)
for name, run in RUNS.items():
    model = Teacher(*run["switches"]).to(DEVICE)
    print(name, "y", tuple(model(batch, dist, bearing)["y"].shape), "| parameters:", sum(p.numel() for p in model.parameters()))

## A5. Training
**Loss** = Huber error of the forecast + pinball loss for the q10 / q90 band (only where the true PM exists):
$$\text{pinball}_\tau(y, q) = \max(\tau(y-q),\ (\tau-1)(y-q))$$
Plain loop: AdamW, learning rate halves when validation stops improving, early stopping, best epoch kept.
**Mixed precision** (`torch.autocast`): the heavy maths runs in float16 on the GPU; the `GradScaler` scales the
loss up so tiny float16 gradients do not become 0.

In [ ]:
# loss: Huber on the forecast + 0.5 x pinball on the band, ignoring missing targets
def loss_fn(out, b):
    mask = b["y_mask"]
    n = mask.sum().clamp_min(1)
    loss = (F.huber_loss(out["y"].float(), b["y"], reduction="none") * mask).sum() / n
    for q, tau in ((out["q10"].float(), 0.1), (out["q90"].float(), 0.9)):
        diff = b["y"] - q
        loss = loss + SETTINGS["pinball_weight"] * (torch.maximum(tau * diff, (tau - 1) * diff) * mask).sum() / n
    return loss

In [ ]:
# run a model on a list of samples and keep what the benchmarks need (as numpy arrays)
@torch.no_grad()
def predict(model, pairs, idx, batch_size=64):
    model.eval()
    idx_t = torch.tensor(idx, device=DEVICE)
    dist_p, bearing_p = geometry(lat[idx], lon[idx])
    keep = {k: [] for k in ("y_hat", "q10", "q90", "y", "y_mask", "base_pm")}
    for k in range(0, len(pairs), batch_size):
        b = make_batch(pairs[k:k + batch_size], idx_t)
        with torch.autocast(DEVICE, dtype=torch.float16, enabled=USE_AMP):
            out = model(b, dist_p, bearing_p)
        for name, value in (("y_hat", out["y"]), ("q10", out["q10"]), ("q90", out["q90"]),
                            ("y", b["y"]), ("y_mask", b["y_mask"]), ("base_pm", b["base_pm"])):
            keep[name].append(value.float().cpu().numpy())
    result = {k: np.concatenate(v) for k, v in keep.items()}
    result["month"] = MONTH[pairs[:, 1].cpu().numpy()]
    return result

In [ ]:
# validation score for early stopping: mean RMSE of the ln change over the 12 outputs
def val_score(model, pairs):
    p = predict(model, pairs, train_idx)
    se = ((p["y_hat"] - p["y"]) ** 2 * p["y_mask"]).sum((0, 1))
    return float(np.sqrt(se / np.maximum(p["y_mask"].sum((0, 1)), 1)).mean())

In [ ]:
# train one run (each epoch uses a fresh random sample of training hours); returns model, history, best val score
def train_teacher(run, pairs):
    seed = RUNS[run]["seed"]
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = Teacher(*RUNS[run]["switches"]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=SETTINGS["lr"], weight_decay=SETTINGS["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=2, factor=0.5)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    val_list = val_pairs[: (64 if SMOKE else 4000)]
    best, best_epoch, best_state, history = float("inf"), 0, None, []
    for epoch in range(SETTINGS["max_epochs"]):
        model.train()
        start = time.time()
        chosen = rng.choice(len(pairs), size=min(SETTINGS["samples_per_epoch"], len(pairs)), replace=False)
        for k in range(0, len(chosen), SETTINGS["batch"]):
            b = make_batch(pairs[chosen[k:k + SETTINGS["batch"]]], TRAIN_IDX, SETTINGS["station_dropout"])
            with torch.autocast(DEVICE, dtype=torch.float16, enabled=USE_AMP):
                loss = loss_fn(model(b, dist, bearing), b)
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)                                   # clip the real gradients, not the scaled ones
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
        val_rmse = round(val_score(model, val_list), 4)
        scheduler.step(val_rmse)
        history.append(dict(run=run, epoch=epoch, val_rmse=val_rmse, seconds=round(time.time() - start)))
        print(history[-1])
        if val_rmse < best:
            best, best_epoch, best_state = val_rmse, epoch, copy.deepcopy(model.state_dict())
        if epoch - best_epoch >= SETTINGS["patience"]:
            break
    model.load_state_dict(best_state)                              # back to the best epoch
    return model, pd.DataFrame(history), best

## A6. Benchmarks
Every run is scored on the test period (2025 – Sep 2026), separately at the 27 training stations ("seen") and at
the 5 stations it never saw ("unseen"), for each of the 12 outputs and each season:

| metric | formula | meaning |
|---|---|---|
| RMSE, MAE (ln) | on $y = \ln(1+PM_{t+h}) - \ln(1+PM_t)$ | error of the predicted change (what the model learns) |
| R² | $1 - \sum(\hat y - y)^2 / \sum(y - \bar y)^2$ | share of the change it explains |
| skill % | $100\,(1 - RMSE / RMSE_{persistence})$ | better than "PM stays the same"? (0 = no better) |
| RMSE, MAE (µg/m³) | $PM = (1+PM_t)\,e^{y} - 1$ | error in real units (with the true $PM_t$ now) |
| category accuracy % | same CPCB band (Good … Severe) for forecast and truth | does it get the AQI colour right? (hourly value, 24-h breakpoints: an indicator) |
| band coverage % | share of truths between q10 and q90 | the uncertainty band; should be ≈ 80 % |

Seasons (IST months): winter Dec–Feb, summer Mar–May, monsoon Jun–Sep, post-monsoon Oct–Nov.

In [ ]:
# CPCB band tops (24-h breakpoints) and the season of every month
BAND_TOPS = {"pm25": [30, 60, 90, 120, 250], "pm10": [50, 100, 250, 350, 430]}
SEASON_OF_MONTH = np.array(["", "winter", "winter", "summer", "summer", "summer", "monsoon", "monsoon",
                            "monsoon", "monsoon", "post-monsoon", "post-monsoon", "winter"])
OUTPUT_NAMES = [f"{p}_h{h}" for p in ("pm25", "pm10") for h in range(1, H + 1)]

In [ ]:
# all benchmarks of one output (1-D arrays of the samples where the truth exists)
def scores(y_hat, y, base_pm, pollutant, q10=None, q90=None):
    pm_true, pm_hat = (1 + base_pm) * np.exp(y) - 1, (1 + base_pm) * np.exp(y_hat) - 1
    band = lambda pm: np.digitize(pm, BAND_TOPS[pollutant], right=True)
    rmse = np.sqrt(np.mean((y_hat - y) ** 2))
    return {"n": len(y), "rmse_ln": rmse, "mae_ln": np.mean(np.abs(y_hat - y)),
            "r2": 1 - np.sum((y_hat - y) ** 2) / np.sum((y - y.mean()) ** 2),
            "skill_%": 100 * (1 - rmse / np.sqrt(np.mean(y ** 2))),
            "rmse_ug": np.sqrt(np.mean((pm_hat - pm_true) ** 2)), "mae_ug": np.mean(np.abs(pm_hat - pm_true)),
            "category_acc_%": 100 * np.mean(band(pm_hat) == band(pm_true)),
            "band_coverage_%": np.nan if q10 is None else 100 * np.mean((q10 <= y) & (y <= q90))}

In [ ]:
# benchmark table of one run: one row per output x season; `cols` = which station columns to score
def benchmark(name, p, stations, cols=slice(None)):
    rows = []
    for o, output in enumerate(OUTPUT_NAMES):
        ok = p["y_mask"][:, cols, o] > 0
        pick = lambda a: a[:, cols, o][ok]
        y_hat, y, base = pick(p["y_hat"]), pick(p["y"]), p["base_pm"][:, cols, o // H][ok]
        q10, q90 = (pick(p["q10"]), pick(p["q90"])) if p["q10"] is not None else (None, None)
        season = SEASON_OF_MONTH[np.broadcast_to(p["month"][:, None], ok.shape)[ok]]
        for s in ("all", "winter", "summer", "monsoon", "post-monsoon"):
            sel = np.ones(len(y), bool) if s == "all" else season == s
            rows.append({"model": name, "stations": stations, "output": output, "season": s,
                         **scores(y_hat[sel], y[sel], base[sel], output[:4],
                                  None if q10 is None else q10[sel], None if q90 is None else q90[sel])})
    return pd.DataFrame(rows)


# persistence = "PM stays the same": the same samples with a predicted change of 0 (and no band)
def persistence(p):
    return {**p, "y_hat": np.zeros_like(p["y_hat"]), "q10": None, "q90": None}

## A7. Teacher outputs for the student (out-of-fold)
The student must never learn from a teacher that already saw the answer. So:
- the training hours are cut into `folds` blocks; for each block we train a teacher **without** it
  (plus a 30-hour gap on both sides) and let that teacher predict the block;
- validation windows are predicted by the final teacher (trained on all training hours).

For every distillation window (`kd_windows.parquet`) we save:

| output | shape | used for |
|---|---|---|
| forecast, h1-h3 | [6] | S1+: copy the forecast |
| time features, averaged per 2-hour patch | [12, 64] | S2+: $L_{feat}$ |
| attention importance per patch | [12] | S2+: $L_{AFD}$ weights |
| GAT state | [64] | S3+: $L_{spatial}$ |
| ADR state | [64] | S4: $L_{physics}$ |

In [ ]:
# attention [heads, 24, 24] -> how important each 2-hour patch is (sums to 1), Na et al.
def patch_importance(attention):
    looked_at = attention.mean(0).sum(0)                       # how much all hours look at each hour
    w = np.exp((looked_at - looked_at.mean()) / (looked_at.std() + 1e-6))
    return (w / w.sum()).reshape(12, 2).sum(1)

In [ ]:
# run a teacher on the hours of some windows and keep its outputs for those (station, hour) pairs
@torch.no_grad()
def teacher_outputs(model, windows, batch_size=16):
    model.eval()
    result = {k: [] for k in ("window_id", "y_teacher", "feat12", "afd12", "h_spatial", "h_adr")}
    groups = list(windows.groupby(["unit", "row"]))
    for k in range(0, len(groups), batch_size):
        chunk = groups[k:k + batch_size]
        b = make_batch(torch.tensor([key for key, _ in chunk], device=DEVICE), TRAIN_IDX)
        with torch.autocast(DEVICE, dtype=torch.float16, enabled=USE_AMP):
            out = model(b, dist, bearing)
        for i, (_, w) in enumerate(chunk):
            pos = torch.as_tensor(np.searchsorted(train_idx, w.station.to_numpy()), device=DEVICE)
            result["window_id"].append(w.window_id.to_numpy())
            result["y_teacher"].append(out["y"][i, pos][:, [0, 1, 2, 6, 7, 8]].float().cpu().numpy())
            fs = out["feat_seq"][i, pos].float().cpu().numpy()
            result["feat12"].append(fs.reshape(len(w), 12, 2, -1).mean(2).astype(np.float16))
            result["afd12"].append(np.stack([patch_importance(a) for a in out["attention"][i, pos].float().cpu().numpy()]))
            result["h_spatial"].append(out["h_spatial"][i, pos].float().cpu().numpy().astype(np.float16))
            result["h_adr"].append(out["h_adr"][i, pos].float().cpu().numpy().astype(np.float16))
    return {k: np.concatenate(v) for k, v in result.items()}

In [ ]:
# the distillation windows (from 03): station, unit, row (= hour), split
kd = pd.read_parquet(FEAT / "kd_windows.parquet")
if SMOKE:
    kd = kd.groupby("split").sample(200, random_state=0)
kd.split.value_counts()

In [ ]:
# out-of-fold outputs of one run: train `folds` extra teachers, each without one block of training hours
def save_student_outputs(run, final_model):
    hours = train_pairs[:, 1].cpu().numpy()
    parts = []
    for fold, block in enumerate(np.array_split(np.unique(hours), SETTINGS["folds"])):
        outside = ~((hours >= block.min() - 30) & (hours <= block.max() + 30))      # block + 30 h gap left out
        fold_model, _, _ = train_teacher(run, train_pairs[torch.tensor(outside, device=DEVICE)])
        windows = kd[(kd.split == "train") & kd.row.between(block.min(), block.max())]
        if len(windows):
            parts.append(teacher_outputs(fold_model, windows))
    parts.append(teacher_outputs(final_model, kd[kd.split == "val"]))
    outputs = {k: np.concatenate([p[k] for p in parts]) for k in parts[0]}
    np.savez(OUT / f"teacher_{run}.npz", **outputs)
    print(run, "outputs for", len(outputs["window_id"]), "windows")

## A8. Train every run
For each run: train on all training hours → benchmark on the test period → (T0-T3 only) out-of-fold outputs.

In [ ]:
# test samples: at the 27 training stations, and at all 32 stations (the 5 unseen ones are scored separately)
test_pairs, test_all = sample_pairs("test", train_idx), sample_pairs("test", all_idx)
if SMOKE:
    test_pairs, test_all = test_pairs[:64], test_all[:64]
len(test_pairs), len(test_all)

In [ ]:
# the main loop
models, histories, val_scores, benchmarks = {}, [], {}, []
for run in VARIANTS:
    if (time.time() - NOTEBOOK_START) / 3600 > TEACHER_BUDGET_H:
        print("time budget used up, skipping", run)
        continue
    models[run], history, val_scores[run] = train_teacher(run, train_pairs)
    histories.append(history)
    p_seen, p_all = predict(models[run], test_pairs, train_idx), predict(models[run], test_all, all_idx)
    if not benchmarks:                                             # persistence needs no model: score it once
        benchmarks += [benchmark("persistence", persistence(p_seen), "seen"),
                       benchmark("persistence", persistence(p_all), "unseen", HOLDOUT_COLS)]
    benchmarks += [benchmark(run, p_seen, "seen"), benchmark(run, p_all, "unseen", HOLDOUT_COLS)]
    if RUNS[run]["for_student"]:
        save_student_outputs(run, models[run])
    print(f"== {run} done, {(time.time() - NOTEBOOK_START) / 60:.0f} min so far")

## A9. Results and ablation study

In [ ]:
# every benchmark (saved), and the summary: mean over the 12 outputs, whole test period
METRICS = ["rmse_ln", "mae_ln", "r2", "skill_%", "rmse_ug", "mae_ug", "category_acc_%", "band_coverage_%"]
table = pd.concat(benchmarks)
table.to_csv(OUT / "teacher_benchmarks.csv", index=False)
summary = table[table.season == "all"].groupby(["stations", "model"], sort=False)[METRICS].mean()
summary.to_csv(OUT / "teacher_summary.csv")
summary.round(3)

In [ ]:
# ablation 1: add one block at a time (training stations)
summary.loc["seen"].reindex(["persistence", "T0", "T1", "T2", "T3"]).dropna(how="all").round(3)

In [ ]:
# ablation 2: take one block out of the full teacher (T2 = T3 without ADR)
summary.loc["seen"].reindex(["T3", "T3-no-wind", "T2", "T3-no-future"]).dropna(how="all").round(3)

In [ ]:
# is a difference real? T3 trained 3 times from different random starts: mean and spread
seeds = summary.loc["seen"].reindex(["T3", "T3-seed1", "T3-seed2"]).dropna(how="all")
seeds.agg(["mean", "std"]).round(3)

In [ ]:
# the 5 stations no teacher ever saw in training
summary.loc["unseen"].round(3)

In [ ]:
# plot: skill vs persistence by forecast hour, PM2.5, training stations
seen = table[(table.stations == "seen") & (table.season == "all") & (table.model != "persistence")]
for name, part in seen.groupby("model", sort=False):
    plt.plot(range(1, H + 1), part["skill_%"].to_numpy()[:H], marker="o", label=name)
plt.axhline(0, color="k", lw=0.8)
plt.xlabel("hours ahead")
plt.ylabel("skill vs persistence (%)")
plt.title("teacher, PM2.5 (higher = better)")
plt.legend(fontsize=7)
plt.show()

In [ ]:
# plot: skill by season (mean over the 12 outputs, training stations)
by_season = table[(table.stations == "seen") & (table.season != "all") & (table.model != "persistence")]
by_season.groupby(["season", "model"], sort=False)["skill_%"].mean().unstack().plot.bar(figsize=(9, 3.5))
plt.ylabel("skill vs persistence (%)")
plt.title("teacher skill by season")
plt.show()

In [ ]:
# plot: training curves (validation error per epoch)
for name, h in pd.concat(histories).groupby("run", sort=False):
    plt.plot(h.epoch, h.val_rmse, label=name)
plt.xlabel("epoch")
plt.ylabel("validation RMSE (ln change)")
plt.legend(fontsize=7)
plt.show()

## A10. Save the best teacher
Only one teacher is kept: the run with the lowest validation error. It is the one the cloud would run live.
(The other runs exist only for the comparison above; the student needs only the `teacher_T*.npz` outputs.)

In [ ]:
# save the best teacher's weights + what is needed to rebuild it
best_run = min(val_scores, key=val_scores.get)
torch.save(models[best_run].state_dict(), OUT / "teacher_best.pt")
(OUT / "teacher_best.json").write_text(json.dumps(
    {"run": best_run, "switches": RUNS[best_run]["switches"], "val_rmse": val_scores[best_run],
     "settings": SETTINGS, "mean": MEAN.tolist(), "std": STD.tolist(), "stations": stations}, indent=1))
print("best teacher:", best_run, "| validation RMSE", val_scores[best_run])

# Part B · Edge student (GDC-CNN) + distillation from the teacher

**Research question:** how much of the teacher's network knowledge (neighbours, wind, transport physics)
survives in a small student that only sees **one** station and runs on the ESP32?

| variant | the student learns from | loss |
|---|---|---|
| S0 | the true answer only | $L_{pred}$ |
| S1 | + the teacher's forecast | $\lambda L_{pred} + (1-\lambda) L_{out}$ (Shmuel et al.) |
| S2 | + the teacher's time features and attention | + $L_{feat} + L_{AFD}$ (Na et al.) |
| S3 | + the teacher's spatial state (GAT) | + $L_{spatial}$ |
| S4 | + the teacher's physics state (ADR) | + $L_{physics}$ |

S3 needs a teacher with a graph (T1-T3), S4 needs T3. Only the student goes to the ESP32;
the small helper layers used for distillation are thrown away.

**Plan of Part B**
1. data → 2. preprocessing (= what the ESP32 does) → 3. the student → 4. benchmarks → 5. one training function
6. **tuning** (learning rate, width) on S0 → 7. S0 → 8-9. distilled students S1-S4 with every teacher
10. **ablation**: remove one loss from S4, and remove one sensor from S0 → 11. results → 12. export the best student

Every student is trained with 3 random seeds, so each result is a mean ± spread (is a difference real or luck?).

**Output files** (`/kaggle/working`): `student_benchmarks.csv`, `student_summary.csv`, `student_tuning.csv`,
and only the best student: `student_best.keras`, `student_best.tflite`, `student_model.h` (for the ESP32), `student_best.json`.

## B0. Setup
PyTorch is done: free its GPU memory, then let TensorFlow take GPU memory only when it needs it.

In [ ]:
# free the teacher's GPU memory and start TensorFlow
del GX, GM, GW, LOG_PM, models
import gc
gc.collect()
torch.cuda.empty_cache()
import tensorflow as tf
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)
import keras
from keras import layers
TEACHER_DIR = OUT                       # Part A just wrote teacher_T0..T3.npz here
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# settings
SMOKE = not ON_KAGGLE                 # tiny quick run on the laptop
MODEL = "A"                           # "A" = PM sensor works, "B" = PM sensor broken (fallback model)
EPOCHS = 2 if SMOKE else 60
PATIENCE = 8
BATCH = 256
LAMBDA_OUT = 0.5                      # S1: half true answer, half teacher forecast
LAMBDA_PRED = 0.4                     # S2-S4: weight of the true answer (Na et al.)
WARMUP = 10                           # distillation weight grows from 0 to 1 over 10 epochs
TEACHERS = ["T0", "T1", "T2", "T3"]
SEEDS = [0] if SMOKE else [0, 1, 2]   # every student is trained 3 times
STUDENT_BUDGET_H = 11.3              # no new student run after 11.3 h of the session (the export always runs)
# tuning grid, tried on S0; the width d is kept small for the ESP32 (d = 32 -> ~100 KB model)
TUNE_GRID = {"lr": [3e-4, 1e-3, 3e-3], "d": [16, 32] if MODEL == "A" else [32, 64]}
if SMOKE:
    TUNE_GRID = {"lr": [1e-3], "d": [16]}

## B1. Data
The student input of a window is cut from the same grids the teacher uses (one station, last 24 hours).

In [ ]:
# the 3 grids (only what the student needs: features, true PM and the time of every hour)
grids = {u: {k: np.load(FEAT / f"grid_u{u}.npz")[k] for k in ("X", "pm", "ts")} for u in range(3)}
MONTH = pd.to_datetime(grids[0]["ts"], unit="s", utc=True).tz_convert("Asia/Kolkata").month.to_numpy()
FEATURES = ["ln_pm25", "ln_pm10", "ln_pm1", "coarse_ratio", "t", "rh", "p", "wind_u", "wind_v", "rain_frac",
            "ln_lux", "mq7_n", "mq135_n", "voc_n", "nox_n", "hod_sin", "hod_cos", "dow_sin", "dow_cos",
            "doy_sin", "doy_cos", "aqi_now", "pm_mask"]
PM_FEATURES = ["ln_pm25", "ln_pm10", "ln_pm1", "coarse_ratio", "aqi_now", "pm_mask"]
USED = FEATURES if MODEL == "A" else [f for f in FEATURES if f not in PM_FEATURES]
print(MODEL, "uses", len(USED), "features")

In [ ]:
# the windows: which (station, unit, hour) are examples
windows_a = pd.read_parquet(FEAT / "windows_A.parquet")
windows_b = pd.read_parquet(FEAT / "windows_B.parquet")
kd_windows = pd.read_parquet(FEAT / "kd_windows.parquet")      # the windows the teacher made outputs for
EXTRAS = ["last_ln_pm25", "last_ln_pm10", "last_aqi", "hours_lost", "pm25_trend"]
len(windows_a), len(windows_b), len(kd_windows)

**Targets.** Model A: $y_h = \ln(1+PM[t+h]) - \ln(1+PM[t])$.
Model B (PM lost $k$ hours ago): relative to the last good hour, $y_h = \ln(1+PM[t+h]) - \ln(1+PM[t-k])$.

In [ ]:
# cut the 24-hour inputs, the 6 targets, the true PM at the base hour and the month out of the grids
def get_xy(w):
    parts = {k: [] for k in ("x", "y", "base_pm", "month")}
    for u, part in w.groupby("unit", sort=True):
        X, pm = grids[u]["X"], grids[u]["pm"]
        rows, st = part.row.to_numpy(), part.station.to_numpy()
        hours = rows[:, None] + np.arange(-23, 1)[None]                    # [n, 24]
        base = rows - (part.k.to_numpy() if MODEL == "B" else 0)            # hour the change is measured from
        log_pm = np.log1p(pm)
        parts["x"].append(X[hours, st[:, None]][:, :, [FEATURES.index(f) for f in USED]])   # [n, 24, C]
        parts["y"].append(np.concatenate([log_pm[rows + h, st, 0:1] - log_pm[base, st, 0:1] for h in (1, 2, 3)] +
                                         [log_pm[rows + h, st, 1:2] - log_pm[base, st, 1:2] for h in (1, 2, 3)], axis=1))
        parts["base_pm"].append(pm[base, st])                                # [n, 2], only for µg/m³ benchmarks
        parts["month"].append(MONTH[rows])
    order = np.argsort(np.concatenate([part.index.to_numpy() for _, part in w.groupby("unit", sort=True)]))
    return {k: np.concatenate(v)[order] for k, v in parts.items()}

In [ ]:
# the 4 sets: distillation train / val (same windows for every variant), test, test at never-seen stations
pool = windows_a if MODEL == "A" else windows_b
if SMOKE:   # the smoke teacher (04) only made outputs for these 200 + 200 windows
    kd_windows = kd_windows.groupby("split").sample(200, random_state=0)
if MODEL == "A":
    train_w = kd_windows[kd_windows.split == "train"].reset_index(drop=True)
    val_w = kd_windows[kd_windows.split == "val"].reset_index(drop=True)
else:   # Model B: the B windows at the same (station, unit, hour) as the teacher's windows
    keys = ["station", "unit", "row"]
    joined = pool.merge(kd_windows[keys + ["window_id"]], on=keys)
    train_w, val_w = [joined[joined.split == s].reset_index(drop=True) for s in ("train", "val")]
test_w = pool[(pool.split == "test") & ~pool.holdout].sample(150_000, random_state=0).reset_index(drop=True)
unseen_w = pool[(pool.split == "test") & pool.holdout].sample(100_000, random_state=0).reset_index(drop=True)
if SMOKE:
    test_w, unseen_w = test_w.head(3000), unseen_w.head(3000)
len(train_w), len(val_w), len(test_w), len(unseen_w)

## B2. Preprocessing (the ESP32 does exactly this in C before calling the model)
1. **Instance norm:** each window, each feature: $\tilde x = (x - \mu_{24h}) / \sigma_{24h}$ — Delhi winter and summer look alike.
   (Not for rain, time-of-day codes and the PM mask.)
2. **Model B:** add the 5 "last good PM" numbers as constant channels.
3. **Patching:** 2 hours → one patch, so 24 h × C becomes 12 patches × 2C.
4. **Target scale:** the model predicts $y / s$ with $s = \max(\sigma_{24h}(\ln PM), 0.1)$; we multiply back by $s$.

`firmware/check_features.py` checks that the ESP32's C code gives the same numbers.

In [ ]:
# instance norm + patches + target scale
NO_NORM = ["rain_frac", "hod_sin", "hod_cos", "dow_sin", "dow_cos", "doy_sin", "doy_cos", "pm_mask"]


def preprocess(x, extras=None):
    norm = np.array([f not in NO_NORM for f in USED])
    mean, std = x.mean(1, keepdims=True), np.sqrt(x.var(1, keepdims=True) + 1e-5)
    xn = np.where(norm, (x - mean) / std, x)
    if MODEL == "A":
        s25 = np.maximum(std[:, 0, USED.index("ln_pm25")], 0.1)
        s10 = np.maximum(std[:, 0, USED.index("ln_pm10")], 0.1)
        scale = np.repeat(np.stack([s25, s10], axis=1), 3, axis=1)
    else:
        xn = np.concatenate([xn, np.repeat(extras[:, None, :], 24, axis=1)], axis=2)
        scale = np.ones((len(x), 6))
    patches = xn.reshape(len(xn), 12, -1)
    return patches.astype(np.float32), scale.astype(np.float32)

In [ ]:
# build the arrays of every set
def make_set(w):
    d = get_xy(w)
    patches, scale = preprocess(d["x"], w[EXTRAS].to_numpy(np.float32) if MODEL == "B" else None)
    return {"x": patches, "y": d["y"].astype(np.float32), "scale": scale, "base_pm": d["base_pm"], "month": d["month"]}


train, val, test, unseen = [make_set(w) for w in (train_w, val_w, test_w, unseen_w)]
train["x"].shape

## B3. The GDC-CNN student (Na et al. 2026)
```
12 patches -> Dense(d) + position -> [GDC block, dilation 1] -> [dilation 2] -> [dilation 4] -> flatten -> 6 outputs
GDC block:  LayerNorm -> causal dilated depthwise conv (kernel 3) -> Dense(2d) -> GLU -> Dense(d) -> + input
```
GLU (gated linear unit): $\text{GLU}(a, b) = a \odot \sigma(b)$ — the gate decides what passes.
Dilations 1, 2, 4 with kernel 3 let the last patch see all 12 patches. Causal = only looks at the past.

In [ ]:
# learned position code for the 12 patches
class PositionCode(layers.Layer):
    def build(self, shape):
        self.code = self.add_weight(shape=(shape[1], shape[2]), initializer="zeros", name="code")

    def call(self, x):
        return x + self.code

In [ ]:
# gated linear unit: first half x sigmoid(second half)
class GLU(layers.Layer):
    def call(self, x):
        half = x.shape[-1] // 2
        return x[..., :half] * keras.ops.sigmoid(x[..., half:])

In [ ]:
# one GDC block
def gdc_block(x, d, dilation):
    h = layers.LayerNormalization(epsilon=1e-5)(x)
    h = layers.ZeroPadding1D((2 * dilation, 0))(h)                        # causal: pad on the left only
    h = layers.DepthwiseConv1D(3, dilation_rate=dilation)(h)
    h = GLU()(layers.Dense(2 * d)(h))
    h = layers.Dense(d)(h)
    return layers.Add()([x, h])

In [ ]:
# the student (width d); `features` also gives the last block's output F_S [12, d] (needed for distillation)
def build_student(d, batch_size=None):
    inp = keras.Input((12, train["x"].shape[2]), batch_size=batch_size)
    x = PositionCode()(layers.Dense(d)(inp))
    for dilation in (1, 2, 4):
        x = gdc_block(x, d, dilation)
    y = layers.Dense(6)(layers.Flatten()(x))
    return keras.Model(inp, y), keras.Model(inp, [y, x])


build_student(16)[0].summary()

## B4. Benchmarks
Every student is scored on the test period (2025 – Sep 2026) at the training stations ("seen") and at the 5
stations never used for training ("unseen"), per output and per season:

| metric | formula | meaning |
|---|---|---|
| RMSE, MAE (ln) | on $y = \ln(1+PM_{t+h}) - \ln(1+PM_t)$ | error of the predicted change (what the model learns) |
| R² | $1 - \sum(\hat y - y)^2 / \sum(y - \bar y)^2$ | share of the change it explains |
| skill % | $100\,(1 - RMSE / RMSE_{persistence})$ | better than "PM stays the same"? (0 = no better) |
| RMSE, MAE (µg/m³) | $PM = (1+PM_t)\,e^{y} - 1$ | error in real units (with the true $PM_t$ now) |
| category accuracy % | same CPCB band (Good … Severe) for forecast and truth | does it get the AQI colour right? (hourly value, 24-h breakpoints: an indicator) |

Seasons (IST months): winter Dec–Feb, summer Mar–May, monsoon Jun–Sep, post-monsoon Oct–Nov.

In [ ]:
# CPCB band tops (24-h breakpoints), the season of every month, the 6 outputs
BAND_TOPS = {"pm25": [30, 60, 90, 120, 250], "pm10": [50, 100, 250, 350, 430]}
SEASON_OF_MONTH = np.array(["", "winter", "winter", "summer", "summer", "summer", "monsoon", "monsoon",
                            "monsoon", "monsoon", "post-monsoon", "post-monsoon", "winter"])
OUTPUTS = ["pm25_h1", "pm25_h2", "pm25_h3", "pm10_h1", "pm10_h2", "pm10_h3"]

In [ ]:
# all benchmarks of one output (1-D arrays)
def scores(y_hat, y, base_pm, pollutant):
    pm_true, pm_hat = (1 + base_pm) * np.exp(y) - 1, (1 + base_pm) * np.exp(y_hat) - 1
    band = lambda pm: np.digitize(pm, BAND_TOPS[pollutant], right=True)
    rmse = np.sqrt(np.mean((y_hat - y) ** 2))
    return {"n": len(y), "rmse_ln": rmse, "mae_ln": np.mean(np.abs(y_hat - y)),
            "r2": 1 - np.sum((y_hat - y) ** 2) / np.sum((y - y.mean()) ** 2),
            "skill_%": 100 * (1 - rmse / np.sqrt(np.mean(y ** 2))),
            "rmse_ug": np.sqrt(np.mean((pm_hat - pm_true) ** 2)), "mae_ug": np.mean(np.abs(pm_hat - pm_true)),
            "category_acc_%": 100 * np.mean(band(pm_hat) == band(pm_true))}

In [ ]:
# benchmark table of one trained student on one set: one row per output x season
def benchmark(name, seed, prediction, data, stations):
    season = SEASON_OF_MONTH[data["month"]]
    rows = []
    for o, output in enumerate(OUTPUTS):
        for s in ("all", "winter", "summer", "monsoon", "post-monsoon"):
            sel = np.ones(len(season), bool) if s == "all" else season == s
            rows.append({"model": name, "seed": seed, "stations": stations, "output": output, "season": s,
                         **scores(prediction[sel, o], data["y"][sel, o], data["base_pm"][sel, o // 3], output[:4])})
    return pd.DataFrame(rows)

In [ ]:
# the model's forecast (ln change) and the validation score used for early stopping / choosing
def forecast(model, data):
    return model.predict(data["x"], batch_size=4096, verbose=0) * data["scale"]


def val_rmse(model, data=None):
    data = val if data is None else data
    return float(np.sqrt(((forecast(model, data) - data["y"]) ** 2).mean(0)).mean())

## B5. One training function for every student
All loss terms work in the real ln-change space (prediction × $s$), Huber $\ell(a, b)$ everywhere.
S0 uses only $L_{pred}$, so S0 and S1-S4 are trained the same way and the comparison is fair.

| term | formula | meaning |
|---|---|---|
| $L_{pred}$ | $\ell(\hat y, y)$ | match the truth |
| $L_{out}$ | $\ell(\hat y, y_T)$ (only gated outputs) | match the teacher's forecast |
| $L_{feat}$ | $\ell(F_S,\ \psi(F_T))$ | student time features ≈ teacher time features (12 patches) |
| $L_{AFD}$ | $\sum_p a_p\,\ell(F_{S,p}, \psi(F_{T,p}))$ | same, weighted by the teacher's attention $a_p$ |
| $L_{spatial}$ | $\ell(P_s(\bar F_S),\ h_{spatial})$ | student guesses the teacher's neighbour state |
| $L_{physics}$ | $\ell(P_p(\bar F_S),\ h_{adr})$ | student guesses the teacher's physics state |

Total for 2+ distillation terms (Na et al. + our extra terms, $n$ = number of terms, $r$ = warm-up 0→1):
$$L = \lambda_p L_{pred} + \frac{1-\lambda_p}{n}\, r(e) \sum L_{kd}$$
$\psi$, $P_s$, $P_p$ are small Dense layers used only in training.

Speed: the training arrays are copied to the GPU **once**; each step only picks rows there (`tf.gather`).

In [ ]:
# the loss terms of each variant
LOSSES = {"S0": [], "S1": ["out"], "S2": ["out", "feat", "afd"],
          "S3": ["out", "feat", "afd", "spatial"], "S4": ["out", "feat", "afd", "spatial", "physics"]}

In [ ]:
# Huber error
def huber(a, b):
    d = tf.abs(a - b)
    return tf.where(d <= 1.0, 0.5 * d * d, d - 0.5)

In [ ]:
# total loss of one batch for a list of distillation terms
def total_loss(terms, nets, b, gate, warmup):
    y_norm, f_s = nets["features"](b["x"], training=True)
    y_hat = y_norm * b["scale"]
    L = {"pred": tf.reduce_mean(huber(y_hat, b["y"]))}
    if not terms:                                                      # S0
        return L["pred"]
    if "out" in terms:
        L["out"] = tf.reduce_mean(huber(y_hat, b["y_t"]) * gate)
    if terms == ["out"]:                                               # S1 (Shmuel et al.)
        return LAMBDA_OUT * L["pred"] + (1 - LAMBDA_OUT) * L["out"]
    error = huber(f_s, nets["psi"](b["f_t"]))                          # [batch, 12, d]
    if "feat" in terms:
        L["feat"] = tf.reduce_mean(error)
    if "afd" in terms:
        L["afd"] = tf.reduce_mean(tf.reduce_sum(b["afd"][:, :, None] * error, axis=1))
    pooled = tf.reduce_mean(f_s, axis=1)
    if "spatial" in terms:
        L["spatial"] = tf.reduce_mean(huber(nets["p_spatial"](pooled), b["h_spatial"]))
    if "physics" in terms:
        L["physics"] = tf.reduce_mean(huber(nets["p_physics"](pooled), b["h_adr"]))
    kd_terms = [L[t] for t in terms]
    return LAMBDA_PRED * L["pred"] + (1 - LAMBDA_PRED) / len(kd_terms) * warmup * tf.add_n(kd_terms)

In [ ]:
# train one student; early stopping on validation RMSE against the TRUE target. Returns (model, best val RMSE)
def train_student(terms, lr, d, seed, train_set, val_set, teacher=None, gate=None):
    keras.utils.set_random_seed(seed)
    student, features = build_student(d)
    nets = {"features": features}
    trainable = list(student.trainable_variables)
    if terms:                                                          # helper layers, only for training
        d_teacher = teacher["f_t"].shape[-1]
        nets.update(psi=layers.Dense(d), p_spatial=layers.Dense(d_teacher), p_physics=layers.Dense(d_teacher))
        nets["psi"].build((None, d_teacher)), nets["p_spatial"].build((None, d)), nets["p_physics"].build((None, d))
        trainable += [w for k in ("psi", "p_spatial", "p_physics") for w in nets[k].trainable_variables]
    arrays = {k: train_set[k] for k in ("x", "y", "scale")} | (teacher or {})
    on_gpu = {k: tf.constant(v) for k, v in arrays.items()}             # copied to the GPU once
    gate = tf.constant(np.ones(6, np.float32) if gate is None else gate)
    opt = keras.optimizers.Adam(lr)

    @tf.function
    def step(rows, warmup):
        b = {k: tf.gather(v, rows) for k, v in on_gpu.items()}
        with tf.GradientTape() as tape:
            loss = total_loss(terms, nets, b, gate, warmup)
        grads = tape.gradient(loss, trainable)
        opt.apply_gradients([(g, w) for g, w in zip(grads, trainable) if g is not None])

    best, best_weights, wait = np.inf, None, 0
    rng = np.random.default_rng(seed)
    n = len(train_set["y"])
    for epoch in range(EPOCHS):
        order = rng.permutation(n).astype(np.int32)
        warmup = tf.constant(min(1.0, epoch / WARMUP), tf.float32)
        for k in range(0, n, BATCH):
            step(tf.constant(order[k:k + BATCH]), warmup)
        score = val_rmse(student, val_set)
        if score < best:
            best, best_weights, wait = score, student.get_weights(), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    student.set_weights(best_weights)
    return student, best

## B6. Tuning (on S0, seed 0)
A small grid of learning rate × width $d$; the best on the **validation** year (2024) is used for every student.
The test period is never looked at while choosing.

In [ ]:
# try every (lr, d) on S0
tuning = []
for lr in TUNE_GRID["lr"]:
    for d in TUNE_GRID["d"]:
        start = time.time()
        _, score = train_student([], lr, d, 0, train, val)
        tuning.append({"lr": lr, "d": d, "val_rmse": score, "minutes": (time.time() - start) / 60})
        print(tuning[-1])
tuning = pd.DataFrame(tuning).sort_values("val_rmse")
tuning.to_csv(OUT / "student_tuning.csv", index=False)
BEST = tuning.iloc[0][["lr", "d"]].to_dict()
BEST["d"] = int(BEST["d"])
print("chosen:", BEST)
tuning.round(4)

In [ ]:
# every trained student is benchmarked here; the best on validation (among S0-S4) is remembered for export
results, best_student = [], {"val_rmse": np.inf}


def run_student(name, seed, terms=(), teacher=None, gate=None, sets=None, can_export=True):
    if (time.time() - NOTEBOOK_START) / 3600 > STUDENT_BUDGET_H:
        print("time budget used up, skipping", name, seed)
        return None
    tr, va, te, un = sets or (train, val, test, unseen)
    model, score = train_student(list(terms), BEST["lr"], BEST["d"], seed, tr, va, teacher, gate)
    results.append(benchmark(name, seed, forecast(model, te), te, "seen"))
    results.append(benchmark(name, seed, forecast(model, un), un, "unseen"))
    if can_export and score < best_student["val_rmse"]:
        best_student.update(name=name, seed=seed, val_rmse=score, weights=model.get_weights())
    print(f"{name} seed {seed}: val RMSE {score:.4f}")
    return model

## B7. S0: the student alone (and persistence, the "PM stays the same" baseline)

In [ ]:
# persistence (prediction = no change) and S0 with every seed
for data, stations in ((test, "seen"), (unseen, "unseen")):
    results.append(benchmark("persistence", 0, np.zeros_like(data["y"]), data, stations))
s0 = {seed: run_student("S0", seed) for seed in SEEDS}

## B8. Teacher outputs for the student's windows
- States are z-scored (training statistics) so every loss term has a similar size.
- Model B: the teacher forecast is from hour $t$; we shift it to "relative to the last good hour":
  $y_T' = y_T + \ln(1+PM_t) - \ln(1+PM_{t-k})$.
- **Precondition (Shmuel et al.):** copy the teacher's forecast only for outputs where it beats S0 on validation.

In [ ]:
# load one teacher's outputs and line them up with our train / val windows
def load_teacher(name):
    path = next(TEACHER_DIR.rglob(f"teacher_{name}.npz"), None)
    if path is None:
        return None
    t = dict(np.load(path))
    position = pd.Series(np.arange(len(t["window_id"])), index=t["window_id"])
    arrays = {}
    for split, w, data in (("train", train_w, train), ("val", val_w, val)):
        i = position.reindex(w.window_id).to_numpy()
        assert not np.isnan(i).any(), "teacher has no output for some windows"
        i = i.astype(int)
        y_t = t["y_teacher"][i].astype(np.float32)
        if MODEL == "B":
            u_rows = [(u, r, s, k) for u, r, s, k in zip(w.unit, w.row, w.station, w.k)]
            shift = np.array([[np.log1p(grids[u]["pm"][r, s, p]) - np.log1p(grids[u]["pm"][r - k, s, p]) for p in (0, 1)]
                              for u, r, s, k in u_rows], np.float32)
            y_t = y_t + np.repeat(shift, 3, axis=1)
        arrays[split] = {"y_t": y_t, "f_t": t["feat12"][i].astype(np.float32), "afd": t["afd12"][i].astype(np.float32),
                         "h_spatial": t["h_spatial"][i].astype(np.float32), "h_adr": t["h_adr"][i].astype(np.float32)}
    for key in ("f_t", "h_spatial", "h_adr"):                         # z-score with training statistics
        axes = (0, 1) if key == "f_t" else 0
        mu, sd = arrays["train"][key].mean(axes), arrays["train"][key].std(axes) + 1e-6
        for split in arrays:
            arrays[split][key] = (arrays[split][key] - mu) / sd
    teacher_val_rmse = np.sqrt(((arrays["val"]["y_t"] - val["y"]) ** 2).mean(0))
    s0_val_rmse = np.sqrt(((forecast(s0[SEEDS[0]], val) - val["y"]) ** 2).mean(0))
    gate = (teacher_val_rmse < s0_val_rmse).astype(np.float32)
    return arrays["train"], gate

## B9. Distilled students S1-S4 with every teacher

In [ ]:
# which teacher can be used for which variant
ALLOWED = {"S1": TEACHERS, "S2": TEACHERS, "S3": ["T1", "T2", "T3"], "S4": ["T3"]}
if MODEL == "B":
    ALLOWED = {"S1": TEACHERS}            # plan: output distillation first for Model B

In [ ]:
# train S1-S4 with every teacher and every seed
teacher_data = {}
for teacher in TEACHERS:
    loaded = load_teacher(teacher)
    if loaded is None:
        print("no outputs for", teacher)
        continue
    teacher_data[teacher] = loaded
    print(teacher, "beats S0 on", [o for o, g in zip(OUTPUTS, loaded[1]) if g])
    for variant, allowed in ALLOWED.items():
        if teacher in allowed:
            for seed in SEEDS:
                run_student(f"{variant}-{teacher}", seed, LOSSES[variant], *loaded)

## B10. Ablation study
**10a. Remove one loss from the full student (S4 with T3).** S4 without $L_{physics}$ is S3-T3, already trained.

**10b. Remove one sensor from S0.** The sensor's channels are set to 0 (as if it were not on the box).
This tells us which sensors the ESP32 node really needs. These models are never exported.

In [ ]:
# 10a: S4-T3 without one loss term
REMOVE_ONE = {"S4-T3-no-out": ["feat", "afd", "spatial", "physics"],
              "S4-T3-no-feat": ["out", "spatial", "physics"],
              "S4-T3-no-spatial": ["out", "feat", "afd", "physics"]}
if MODEL == "A" and "T3" in teacher_data:
    for name, terms in REMOVE_ONE.items():
        for seed in SEEDS:
            run_student(name, seed, terms, *teacher_data["T3"])

In [ ]:
# 10b: S0 without one sensor (its feature channels set to 0 in every set)
SENSOR_GROUPS = {"S0-no-MQ7": ["mq7_n"], "S0-no-MQ135": ["mq135_n"], "S0-no-SGP41": ["voc_n", "nox_n"],
                 "S0-no-BH1750": ["ln_lux"], "S0-no-fans": ["wind_u", "wind_v"], "S0-no-rain": ["rain_frac"],
                 "S0-no-BMP280": ["p"]}


def without(data, names):
    per_hour = data["x"].shape[2] // 2                                 # a patch = 2 hours side by side
    cols = [USED.index(f) for f in names]
    x = data["x"].copy()
    x[:, :, cols] = 0
    x[:, :, [per_hour + c for c in cols]] = 0
    return {**data, "x": x}


for name, names in SENSOR_GROUPS.items():
    sets = [without(s, names) for s in (train, val, test, unseen)]
    for seed in SEEDS:
        run_student(name, seed, sets=sets, can_export=False)

## B11. Results
Each number = mean over the 6 outputs, then mean over the 3 seeds; `skill_±` = spread (std) over the seeds.
A difference smaller than about 2 × the spread may just be luck.

In [ ]:
# every benchmark (saved) and the summary
METRICS = ["rmse_ln", "mae_ln", "r2", "skill_%", "rmse_ug", "mae_ug", "category_acc_%"]
table = pd.concat(results)
table.to_csv(OUT / "student_benchmarks.csv", index=False)
per_run = table[table.season == "all"].groupby(["stations", "model", "seed"], sort=False)[METRICS].mean()
summary = per_run.groupby(level=["stations", "model"], sort=False).mean()
summary["skill_±"] = per_run["skill_%"].groupby(level=["stations", "model"], sort=False).std()
s0_skill = summary["skill_%"].xs("S0", level="model")                  # S0's skill, per station group
summary["skill_vs_S0"] = summary["skill_%"] - s0_skill.reindex(summary.index.get_level_values("stations")).to_numpy()
summary.to_csv(OUT / "student_summary.csv")
summary.loc["seen"].round(3)

In [ ]:
# distillation: does learning from the teacher help? (training stations)
summary.loc["seen"].filter(regex=r"^(persistence|S0|S\d-T\d)$", axis=0).round(3)

In [ ]:
# ablation 10a: take one loss out of S4-T3 (S3-T3 = S4-T3 without L_physics)
summary.loc["seen"].reindex(["S4-T3", "S4-T3-no-out", "S4-T3-no-feat", "S4-T3-no-spatial", "S3-T3"]).dropna(how="all").round(3)

In [ ]:
# ablation 10b: which sensors matter? (negative skill_vs_S0 = the model is worse without that sensor)
sensors = summary.loc["seen"].reindex(["S0"] + list(SENSOR_GROUPS)).dropna(how="all")
sensors["skill_vs_S0"].iloc[1:].sort_values().plot.barh(figsize=(6, 3), title="skill lost without each sensor (%)")
plt.axvline(0, color="k", lw=0.8)
plt.show()
sensors.round(3)

In [ ]:
# the 5 stations never used for training
summary.loc["unseen"].round(3)

In [ ]:
# plot: skill vs persistence by forecast hour, PM2.5 (persistence, S0 and the distilled students, seed mean)
seen = table[(table.stations == "seen") & (table.season == "all") & table.model.str.match(r"^(S0|S\d-T\d)$")]
for name, part in seen.groupby("model", sort=False):
    plt.plot([1, 2, 3], part.groupby("output", sort=False)["skill_%"].mean().to_numpy()[:3], marker="o", label=name)
plt.axhline(0, color="k", lw=0.8)
plt.xlabel("hours ahead")
plt.ylabel("skill vs persistence (%)")
plt.title(f"Model {MODEL}: student, PM2.5 (higher = better)")
plt.legend(fontsize=7, ncol=3)
plt.show()

In [ ]:
# plot: skill by season for S0 and the distilled students (training stations)
by_season = table[(table.stations == "seen") & (table.season != "all") & table.model.str.match(r"^(S0|S\d-T\d)$")]
by_season.groupby(["season", "model"], sort=False)["skill_%"].mean().unstack().plot.bar(figsize=(10, 3.5), legend=False)
plt.ylabel("skill vs persistence (%)")
plt.title("student skill by season (one bar per student)")
plt.show()

## B12. Export the best student for the ESP32 (TFLite, FP32)
Only one student is kept: the lowest validation error among S0-S4 (never the sensor-ablation models).
Rebuilt with batch size 1 (fixed shapes for TFLite Micro), converted, and checked: it must give the
same numbers as the Keras model.

In [ ]:
# rebuild the best student and save it (Keras + TFLite)
print("best student:", best_student["name"], "seed", best_student["seed"], f"val RMSE {best_student['val_rmse']:.4f}")
export_model, _ = build_student(BEST["d"], batch_size=1)
export_model.set_weights(best_student["weights"])
export_model.save(OUT / "student_best.keras")
tflite_bytes = tf.lite.TFLiteConverter.from_keras_model(export_model).convert()
(OUT / "student_best.tflite").write_bytes(tflite_bytes)
print(f"{len(tflite_bytes) / 1024:.1f} KB, {export_model.count_params()} parameters")

In [ ]:
# the same model as a C array: the ESP32 sketch includes it (kaggle_run.py download copies it
# to firmware/aqi_node/student_model.h, right beside aqi_node.ino)
lines = [", ".join(f"0x{b:02x}" for b in tflite_bytes[i:i + 20]) for i in range(0, len(tflite_bytes), 20)]
header = [f"// student {best_student['name']} (seed {best_student['seed']}), model {MODEL}, made by notebook 04 (Part B).",
          "// Do not edit: re-export instead.",
          f'#define STUDENT_NAME "{best_student["name"]}"',
          f"#define STUDENT_MODEL_{MODEL}",
          f"#define STUDENT_FEATURES {len(USED)}",
          "alignas(16) const unsigned char student_model[] = {",
          ",\n".join(lines),
          "};",
          f"const unsigned int student_model_len = {len(tflite_bytes)};"]
(OUT / "student_model.h").write_text("\n".join(header) + "\n")

In [ ]:
# check: TFLite gives the same output as Keras, and how long one forecast takes (Kaggle CPU, not the ESP32)
interpreter = tf.lite.Interpreter(model_content=tflite_bytes)
interpreter.allocate_tensors()
inp, out = interpreter.get_input_details()[0]["index"], interpreter.get_output_details()[0]["index"]
worst, start = 0.0, time.time()
for x in val["x"][:200]:
    interpreter.set_tensor(inp, x[None])
    interpreter.invoke()
    worst = max(worst, float(np.abs(interpreter.get_tensor(out) - export_model(x[None]).numpy()).max()))
print(f"largest difference TFLite vs Keras: {worst:.1e}")
assert worst < 1e-4

In [ ]:
# what the ESP32 sketch and the report need to know about the exported student
info = {"name": best_student["name"], "seed": best_student["seed"], "val_rmse": best_student["val_rmse"],
        "model": MODEL, "lr": BEST["lr"], "d": BEST["d"], "features": USED, "no_norm": NO_NORM,
        "tflite_kb": round(len(tflite_bytes) / 1024, 1), "parameters": int(export_model.count_params())}
(OUT / "student_best.json").write_text(json.dumps(info, indent=1))
info